<h1 style="color:red;">Préparer data pour la classification par variete</h1>

In [6]:
import os
import zipfile
from typing import List
import time
from pathlib import Path
import shutil


In [7]:
# data paths : ===================================================================
ZIP_DIR = "../data/data_zip"
EXTRACT_DIR =  "../data/dataset_extrait/"
LIST_FILES_ZIP = ["Boufagous2", "bouisthami", "Boufagous", "Boufagous3", "Boumajhoul", "Boumajhoul2", "kholt"]

os.makedirs(EXTRACT_DIR, exist_ok=True)
# ===================================================================
EXTENSIONS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")
# ===================================================================

* Afficher les fichiers ziper :

In [8]:
# ===================================================================
def zip_files(ZIP_DIR: str = ZIP_DIR):
    print("ZIP trouvés :")
    for f in os.listdir(ZIP_DIR):
        if f.endswith(".zip"):
            print(" -", f)
            
# ===================================================================
zip_files(ZIP_DIR)


ZIP trouvés :
 - Annotations.zip
 - Boufagous.zip
 - Boufagous2.zip
 - Boufagous3.zip
 - bouisthami.zip
 - Boumajhoul.zip
 - Boumajhoul2.zip
 - kholt.zip


* Compter les images dans chaque fichier zipper :

In [9]:
# ===================================================================
def count_images_in_zip_files(list_files_zip : List[str] = LIST_FILES_ZIP, zip_directory : str = ZIP_DIR, extensions: set = EXTENSIONS):
  total = 0
  for doc in list_files_zip:
    zip_path = f"{zip_directory}/{doc}.zip"

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        files = zip_ref.namelist()

    images = [f for f in files if f.lower().endswith(extensions)]
    total += len(images)
    print(f"\nImages dans le {doc}.zip : {len(images)}")

  print(f"Total images ziper : {total}")
  
# ===================================================================
count_images_in_zip_files(list_files_zip=LIST_FILES_ZIP, zip_directory=ZIP_DIR)


Images dans le Boufagous2.zip : 1174

Images dans le bouisthami.zip : 360

Images dans le Boufagous.zip : 1476

Images dans le Boufagous3.zip : 1675

Images dans le Boumajhoul.zip : 877

Images dans le Boumajhoul2.zip : 1768

Images dans le kholt.zip : 1761
Total images ziper : 9091


* Extraire les images :

In [10]:
# ===================================================================
def extract_files_zip(zip_directory:str = ZIP_DIR, extract_directory: str = EXTRACT_DIR):
    start_time_extract = time.time()
    for zip_file in os.listdir(zip_directory):
        if zip_file.endswith(".zip"):
            class_name = zip_file.replace(".zip", "")

            extract_path = os.path.join(extract_directory, class_name)
            os.makedirs(extract_path, exist_ok=True)

            zip_path = os.path.join(zip_directory, zip_file)
            print(f"📦 Extraction de {zip_file} → {extract_path}")

            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
    end_time_extract = time.time()
    print(f"    🕐 DUREE : {(end_time_extract-start_time_extract)/60:.2f} minutes")
    
# ===================================================================

extract_files_zip(zip_directory = ZIP_DIR, extract_directory =  EXTRACT_DIR)
print("✅ Extraction terminée (structure conservée, aucune perte)")


📦 Extraction de Annotations.zip → ../data/dataset_extrait/Annotations
📦 Extraction de Boufagous.zip → ../data/dataset_extrait/Boufagous
📦 Extraction de Boufagous2.zip → ../data/dataset_extrait/Boufagous2
📦 Extraction de Boufagous3.zip → ../data/dataset_extrait/Boufagous3
📦 Extraction de bouisthami.zip → ../data/dataset_extrait/bouisthami
📦 Extraction de Boumajhoul.zip → ../data/dataset_extrait/Boumajhoul
📦 Extraction de Boumajhoul2.zip → ../data/dataset_extrait/Boumajhoul2
📦 Extraction de kholt.zip → ../data/dataset_extrait/kholt
    🕐 DUREE : 3.75 minutes
✅ Extraction terminée (structure conservée, aucune perte)


* Structure de dossier de data unziper

In [11]:
# ===================================================================
def print_tree(dir_path, indent=""):
    for item in sorted(os.listdir(dir_path)):
        path = os.path.join(dir_path, item)
        if os.path.isdir(path):
            print(f"{indent}📁 {item}")
            print_tree(path, indent + "    ")

# ===================================================================
print_tree(EXTRACT_DIR)

📁 Annotations
    📁 Annotations
        📁 classification
        📁 object detection
            📁 date fruits detection
            📁 date fruits maturity
            📁 date fruits varieties
📁 Boufagous
    📁 Boufagous
        📁 S1
        📁 S2
📁 Boufagous2
    📁 Boufagous2
        📁 S4
📁 Boufagous3
    📁 Boufagous3
        📁 S3
📁 Boumajhoul
    📁 Boumajhoul
        📁 S1
        📁 S2
📁 Boumajhoul2
    📁 Boumajhoul2
        📁 S3
        📁 S4
📁 bouisthami
    📁 bouisthami
        📁 S1
        📁 S2
        📁 S3
        📁 S4
📁 kholt
    📁 kholt
        📁 S1
        📁 S2
        📁 S3
        📁 S4


* Compter les images dans chaque fichier unzipper :

In [12]:


# ===================================================================
def count_images(folder, images_extensions: list[str] = list(EXTENSIONS) ):
    count = 0
    for ext in images_extensions:
        count += len(list(Path(folder).rglob(f"*{ext}")))
    return count

# ===================================================================

def count_img_in_each_file(dataset_dir: str = EXTRACT_DIR):
    for datte_type in sorted(os.listdir(dataset_dir)):
        datte_path = os.path.join(dataset_dir, datte_type)
        
        if not os.path.isdir(datte_path):
            continue

        print(f"\n🌴 Type : {datte_type}")

        found_stage = False

        for stage_dir in Path(datte_path).rglob("*"):
            if stage_dir.is_dir() and stage_dir.name.upper().startswith("S"):
                nb_images = count_images(stage_dir, images_extensions = list(EXTENSIONS))
                print(f"   └── {stage_dir.name} : {nb_images} images")
                found_stage = True

        if not found_stage:
            print("   ⚠️ Aucun dossier de stage (S1, S2, ...) trouvé")


# ===================================================================
count_img_in_each_file(dataset_dir=EXTRACT_DIR)


🌴 Type : Annotations
   ⚠️ Aucun dossier de stage (S1, S2, ...) trouvé

🌴 Type : Boufagous
   └── S1 : 1826 images
   └── S2 : 1126 images

🌴 Type : Boufagous2
   └── S4 : 2348 images

🌴 Type : Boufagous3
   └── S3 : 3350 images

🌴 Type : Boumajhoul
   └── S1 : 608 images
   └── S2 : 1146 images

🌴 Type : Boumajhoul2
   └── S3 : 2690 images
   └── S4 : 846 images

🌴 Type : bouisthami
   └── S1 : 190 images
   └── S2 : 274 images
   └── S3 : 92 images
   └── S4 : 164 images

🌴 Type : kholt
   └── S1 : 278 images
   └── S2 : 828 images
   └── S3 : 1298 images
   └── S4 : 1118 images


In [13]:
print_tree(EXTRACT_DIR)

📁 Annotations
    📁 Annotations
        📁 classification
        📁 object detection
            📁 date fruits detection
            📁 date fruits maturity
            📁 date fruits varieties
📁 Boufagous
    📁 Boufagous
        📁 S1
        📁 S2
📁 Boufagous2
    📁 Boufagous2
        📁 S4
📁 Boufagous3
    📁 Boufagous3
        📁 S3
📁 Boumajhoul
    📁 Boumajhoul
        📁 S1
        📁 S2
📁 Boumajhoul2
    📁 Boumajhoul2
        📁 S3
        📁 S4
📁 bouisthami
    📁 bouisthami
        📁 S1
        📁 S2
        📁 S3
        📁 S4
📁 kholt
    📁 kholt
        📁 S1
        📁 S2
        📁 S3
        📁 S4


* Explorer les extensions des images dans la data :

In [14]:
# ===================================================================
def verify_extensions(directory_dataset: str = EXTRACT_DIR):
    extensions = set()
    for file in Path(directory_dataset).rglob("*"):
        if file.is_file():
            extensions.add(file.suffix)

    print("Extensions trouvées dans le dataset :")
    for ext in sorted(extensions):
        print(ext)
        
# ===================================================================
verify_extensions(directory_dataset = EXTRACT_DIR)


Extensions trouvées dans le dataset :
.JPG
.jpeg
.jpg
.txt


* organiser les images par types et par stages avec le comptage des iamges dans chaque groupe:

In [15]:

# ===================================================================
FINAL_DATASET = "../data/dataset_type_stages"

os.makedirs(FINAL_DATASET, exist_ok=True)

# ===================================================================
MERGE_RULES = {
    "Boufagous": ["Boufagous", "Boufagous2", "Boufagous3"],
    "Boumajhoul": ["Boumajhoul", "Boumajhoul2"],
    "kholt": ["kholt"],
    "bouisthami": ["bouisthami"]
}

STAGES = ["S1", "S2", "S3", "S4"]
# ===================================================================


In [16]:
# ===================================================================
def create_directory(path):
    """Créer un dossier s'il n'existe pas"""
    os.makedirs(path, exist_ok=True)


# ===================================================================
def copy_images_from_stage(src_root, src_type, stage, final_stage_dir, extensions):

    total_images = 0

    stage_path = os.path.join(src_root, src_type, stage)

    if not os.path.isdir(stage_path):
        return 0

    for file in Path(stage_path).iterdir():

        if file.is_file() and file.suffix.lower() in extensions:

            shutil.copy(
                file,
                os.path.join(final_stage_dir, file.name)
            )

            total_images += 1

    return total_images


# ===================================================================
def process_stage(final_class_dir, stage, source_types, extract_dir, extensions):
    """Traiter un stage (S1, S2, ...)"""

    final_stage_dir = os.path.join(final_class_dir, stage)
    create_directory(final_stage_dir)

    total_stage_images = 0

    for src_type in source_types:

        src_root = os.path.join(extract_dir, src_type)

        if not os.path.isdir(src_root):
            continue

        total_stage_images += copy_images_from_stage(
            src_root,
            src_type,
            stage,
            final_stage_dir,
            extensions
        )

    if total_stage_images > 0:
        print(f"   └── {stage} : {total_stage_images} images")


# ===================================================================
def process_class(final_class, source_types, stages, extract_dir, final_dataset, extensions):
    """Fusionner toutes les données d'une classe finale"""

    print(f"\n🌴 Fusion → {final_class}")

    final_class_dir = os.path.join(final_dataset, final_class)
    create_directory(final_class_dir)

    for stage in stages:
        process_stage(
            final_class_dir,
            stage,
            source_types,
            extract_dir,
            extensions
        )


# ===================================================================
def merge_dataset(merge_rules, stages, extract_dir, final_dataset, extensions):
    """Fonction principale de fusion"""

    for final_class, source_types in merge_rules.items():

        process_class(
            final_class,
            source_types,
            stages,
            extract_dir,
            final_dataset,
            extensions
        )

    print("\n✅ Fusion terminée sans renommage et sans perte")


# ===================================================================

In [17]:
merge_dataset(
    MERGE_RULES,
    STAGES,
    EXTRACT_DIR,
    FINAL_DATASET,
    EXTENSIONS
)


🌴 Fusion → Boufagous
   └── S1 : 913 images
   └── S2 : 563 images
   └── S3 : 1675 images
   └── S4 : 1174 images

🌴 Fusion → Boumajhoul
   └── S1 : 304 images
   └── S2 : 573 images
   └── S3 : 1345 images
   └── S4 : 423 images

🌴 Fusion → kholt
   └── S1 : 139 images
   └── S2 : 414 images
   └── S3 : 649 images
   └── S4 : 559 images

🌴 Fusion → bouisthami
   └── S1 : 95 images
   └── S2 : 137 images
   └── S3 : 46 images
   └── S4 : 82 images

✅ Fusion terminée sans renommage et sans perte


* Total des images pour chaque types :

In [18]:
# ===================================================================
def count_images_by_type(dataset, extensions):

    total_dataset = 0

    for cls in os.listdir(dataset):

        class_path = os.path.join(dataset, cls)

        if not os.path.isdir(class_path):
            continue

        images = [
            f for f in Path(class_path).rglob("*")
            if f.is_file() and f.suffix.lower() in extensions
        ]

        count = len(images)

        print(f"{cls} : {count} images")

        total_dataset += count

    print("\n==============================")
    print(f"Total dataset : {total_dataset} images")
    
    
# ===================================================================
count_images_by_type(FINAL_DATASET, EXTENSIONS)

Boufagous : 4325 images
bouisthami : 360 images
Boumajhoul : 2645 images
kholt : 1761 images

Total dataset : 9091 images


In [19]:
print_tree(FINAL_DATASET)

📁 Boufagous
    📁 S1
    📁 S2
    📁 S3
    📁 S4
📁 Boumajhoul
    📁 S1
    📁 S2
    📁 S3
    📁 S4
📁 bouisthami
    📁 S1
    📁 S2
    📁 S3
    📁 S4
📁 kholt
    📁 S1
    📁 S2
    📁 S3
    📁 S4


* supprimer les dossiers non utilisées

In [20]:

# ===================================================================                   
def delete_paths(paths):
    """
    Supprimer une liste de dossiers.
    """
    for path in paths:

        if os.path.exists(path):
            shutil.rmtree(path)
            print(f"✅ Supprimé : {path}")

        else:
            print(f"⚠️ Dossier introuvable : {path}")
            

# ===================================================================                   
paths_to_delete = [
    # "../dataset",
    EXTRACT_DIR,
]

# ===================================================================                   
delete_paths(paths_to_delete)


✅ Supprimé : ../data/dataset_extrait/


***

* organiser dataset pour la classification pr variete : 

In [21]:
def build_variety_dataset(final_dataset, dataset_variete_dir, extensions):
    """
    Fusionne les images S1..S4 pour créer un dataset de classification par variété.
    """

    image_ext = [ext.lower() for ext in extensions]

    os.makedirs(dataset_variete_dir, exist_ok=True)

    for variety in os.listdir(final_dataset):

        variety_path = os.path.join(final_dataset, variety)

        if not os.path.isdir(variety_path):
            continue

        print(f"\n🌴 Traitement : {variety}")

        target_variety_dir = os.path.join(dataset_variete_dir, variety)
        os.makedirs(target_variety_dir, exist_ok=True)

        img_count = 0

        # parcourir S1, S2, S3, S4
        for stage in os.listdir(variety_path):

            stage_path = os.path.join(variety_path, stage)

            if not os.path.isdir(stage_path):
                continue

            for img in Path(stage_path).iterdir():

                if img.suffix.lower() in image_ext:

                    # éviter écrasement
                    new_name = f"{stage}_{img.name}"

                    shutil.copy(
                        img,
                        os.path.join(target_variety_dir, new_name)
                    )

                    img_count += 1

        print(f"   ✅ {img_count} images fusionnées")

    print("\n🎉 Dataset prêt pour la classification")

In [22]:
DATASET_VARIETE_DIR = "../data/dataset_classification/dataset_variete"

build_variety_dataset(
    FINAL_DATASET,
    DATASET_VARIETE_DIR,
    EXTENSIONS
)


🌴 Traitement : Boufagous
   ✅ 4325 images fusionnées

🌴 Traitement : bouisthami
   ✅ 360 images fusionnées

🌴 Traitement : Boumajhoul
   ✅ 2645 images fusionnées

🌴 Traitement : kholt
   ✅ 1761 images fusionnées

🎉 Dataset prêt pour la classification


In [26]:
print_tree(DATASET_VARIETE_DIR)

📁 Boufagous
📁 Boumajhoul
📁 bouisthami
📁 kholt


***

* organiser dataset pour la classification pr maturite : 

In [29]:
import os
import shutil
from pathlib import Path

def build_maturity_dataset(source_dataset, target_dataset, extensions):
    """
    Organise le dataset pour la classification par maturité (S1,S2,S3,S4)
    """

    image_ext = [ext.lower() for ext in extensions]

    os.makedirs(target_dataset, exist_ok=True)

    total_images = 0

    for variety in os.listdir(source_dataset):

        variety_path = os.path.join(source_dataset, variety)

        if not os.path.isdir(variety_path):
            continue

        print(f"\n🌴 Lecture variété : {variety}")

        for stage in os.listdir(variety_path):

            stage_path = os.path.join(variety_path, stage)

            if not os.path.isdir(stage_path):
                continue

            target_stage_dir = os.path.join(target_dataset, stage)
            os.makedirs(target_stage_dir, exist_ok=True)

            img_count = 0

            for img in Path(stage_path).iterdir():

                if img.suffix.lower() in image_ext:

                    # ajouter la variété pour éviter les doublons
                    new_name = f"{variety}_{img.name}"

                    shutil.copy(
                        img,
                        os.path.join(target_stage_dir, new_name)
                    )

                    img_count += 1
                    total_images += 1

            print(f"   └── {stage} : {img_count} images")

    print("\n🎉 Dataset maturité prêt")
    print(f"📊 Total images : {total_images}")

In [30]:
DATASET_MATURITY_DIR = "../data/dataset_classification/dataset_maturity"

build_maturity_dataset(
    source_dataset=FINAL_DATASET,
    target_dataset=DATASET_MATURITY_DIR,
    extensions=EXTENSIONS
)


🌴 Lecture variété : Boufagous
   └── S1 : 913 images
   └── S2 : 563 images
   └── S3 : 1675 images
   └── S4 : 1174 images

🌴 Lecture variété : bouisthami
   └── S1 : 95 images
   └── S2 : 137 images
   └── S3 : 46 images
   └── S4 : 82 images

🌴 Lecture variété : Boumajhoul
   └── S1 : 304 images
   └── S2 : 573 images
   └── S3 : 1345 images
   └── S4 : 423 images

🌴 Lecture variété : kholt
   └── S1 : 139 images
   └── S2 : 414 images
   └── S3 : 649 images
   └── S4 : 559 images

🎉 Dataset maturité prêt
📊 Total images : 9091


***

* Organiser la data pour la detection avec Yolov8

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path


# ============================================================
def extract_annotations(zip_path, extract_dir):
    """Extraire le fichier zip contenant les annotations"""

    os.makedirs(extract_dir, exist_ok=True)

    print("📦 Décompression du fichier...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print("✅ Extraction terminée dans :", extract_dir)

    print("\n📂 Contenu extrait :")
    for root, dirs, files in os.walk(extract_dir):
        print(root, "→", len(files), "fichiers")


# ============================================================
def copy_annotations(source_dir, target_dir, raw_dir):
    """Copier les fichiers d'annotations vers le dossier labels"""

    os.makedirs(target_dir, exist_ok=True)

    print("\n📂 Copie des annotations...")

    for file in os.listdir(source_dir):

        src_file = os.path.join(source_dir, file)
        dst_file = os.path.join(target_dir, file)

        if os.path.isfile(src_file):
            shutil.copy(src_file, dst_file)

    print("✅ Annotations copiées dans :", target_dir)

    print("🗑️ Suppression du dossier temporaire...")
    shutil.rmtree(raw_dir)

    print("🎉 Nettoyage terminé")


# ============================================================
def merge_images(source_dir, target_dir, image_ext):
    """Fusionner toutes les images dans un seul dossier"""

    os.makedirs(target_dir, exist_ok=True)

    img_count = 0

    print("\n📦 Fusion des images...")

    for root, dirs, files in os.walk(source_dir):

        for file in files:

            if Path(file).suffix.lower() in image_ext:

                src_path = os.path.join(root, file)
                dst_path = os.path.join(target_dir, file)

                if not os.path.exists(dst_path):

                    shutil.copy(src_path, dst_path)
                    img_count += 1

                    if img_count % 100 == 0:
                        print(f"    🔔 {img_count} images copiées")

                else:
                    print(f"⚠️ Image déjà existante ignorée : {file}")

    print(f"\n✅ {img_count} images copiées dans {target_dir}")


# ============================================================
def show_dataset_stats(images_dir, labels_dir):
    """Afficher le nombre d'images et d'annotations"""

    n_images = len(os.listdir(images_dir))
    n_labels = len(os.listdir(labels_dir))

    print("\n📊 Statistiques du dataset")
    print("Images :", n_images)
    print("Labels :", n_labels)

In [ ]:
# chemins
ANNOTATION_ZIP_PATH = "../data/data_zip/Annotations.zip"
ANNOTATION_EXTRACT_DIR = "../data/dataset_raw_annotation"

SOURCE_LABEL_DIR = "../data/dataset_raw_annotation/Annotations/object detection/date fruits detection"
TARGET_LABEL_DIR = "../data/dataset_detection/data_merged/labels"

SOURCE_IMAGE_DIR = "../data/dataset_type_stages"
TARGET_IMAGE_DIR = "../data/dataset_detection/data_merged/images"

IMAGE_EXT = [".jpg", ".jpeg", ".png"]


# pipeline
extract_annotations(ANNOTATION_ZIP_PATH, ANNOTATION_EXTRACT_DIR)

copy_annotations(
    SOURCE_LABEL_DIR,
    TARGET_LABEL_DIR,
    ANNOTATION_EXTRACT_DIR
)

merge_images(
    SOURCE_IMAGE_DIR,
    TARGET_IMAGE_DIR,
    list(EXTENSIONS)
)

show_dataset_stats(
    TARGET_IMAGE_DIR,
    TARGET_LABEL_DIR
)


📦 Fusion des images...
    🔔 10 images copiées
    🔔 20 images copiées
    🔔 30 images copiées
    🔔 40 images copiées
    🔔 50 images copiées
    🔔 60 images copiées
    🔔 70 images copiées
    🔔 80 images copiées
    🔔 90 images copiées
    🔔 100 images copiées
    🔔 110 images copiées
    🔔 120 images copiées
    🔔 130 images copiées
    🔔 140 images copiées
    🔔 150 images copiées
    🔔 160 images copiées
    🔔 170 images copiées
    🔔 180 images copiées
    🔔 190 images copiées
    🔔 200 images copiées
    🔔 210 images copiées
    🔔 220 images copiées
    🔔 230 images copiées
    🔔 240 images copiées
    🔔 250 images copiées
    🔔 260 images copiées
    🔔 270 images copiées
    🔔 280 images copiées
    🔔 290 images copiées
    🔔 300 images copiées
    🔔 310 images copiées
    🔔 320 images copiées
    🔔 330 images copiées
    🔔 340 images copiées
    🔔 350 images copiées
    🔔 360 images copiées
    🔔 370 images copiées
    🔔 380 images copiées
    🔔 390 images copiées
    🔔 400 